<a href="https://colab.research.google.com/github/baothai12082006/CollectDataFinance/blob/main/CollectData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ==============================================================================
# BƯỚC 1: CẤU HÌNH GOOGLE CHROME ĐỂ TỰ ĐỘNG DOWNLOAD FILE KHI CHẠY CODE JS
# ==============================================================================
print("[*] Đang khởi động cấu hình môi trường tải file bằng ép mã Javascript chạy ngầm...")
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update -y
!apt-get install -y google-chrome-stable > /dev/null
!pip install selenium pypdf requests > /dev/null

from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import shutil
import tempfile
import threading
import time
from collections import Counter, defaultdict, deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable, Optional
from urllib.parse import urljoin

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from selenium import webdriver
from selenium.common.exceptions import WebDriverException
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# ------------------------------- Configuration --------------------------------
ROOT_DIR = Path("./Du_Lieu_BCTC_Cac_Cong_Ty")
DRIVE_TARGET_DIR = Path("/content/drive/MyDrive/Du_Lieu_BCTC_Cac_Cong_Ty")
CACHE_FILE = ROOT_DIR / "tasks.json"
LOG_FILE = ROOT_DIR / "cafef_downloader_v2.log"
BANK_TICKERS = ["PVI"]
YEARS = ("2021", "2022", "2023", "2024", "2025")
MIN_FILE_BYTES = 10 * 1024
REQUEST_WORKERS = 6             # requests only; Chrome remains bounded below
MAX_RETRY_ROUNDS = 3
PAGE_WAIT_SECONDS = 15
DOWNLOAD_WAIT_SECONDS = 20
USER_AGENT = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")


@dataclass(frozen=True)
class ReportTask:
    ticker: str
    exchange: str
    year: str
    period: str
    title: str
    filename: str
    sub_path: str
    href: str = ""
    onclick: str = ""
    outer_html: str = ""
    row_text: str = ""
    source_url: str = ""
    link_index: int = 0
    task_id: str = ""

    @property
    def local_path(self) -> Path:
        return ROOT_DIR / self.sub_path / self.filename

    @property
    def drive_path(self) -> Path:
        return DRIVE_TARGET_DIR / self.sub_path / self.filename

    @staticmethod
    def from_dict(data: dict) -> "ReportTask":
        allowed = {k: v for k, v in data.items() if k in ReportTask.__dataclass_fields__}
        return ReportTask(**allowed)


@dataclass
class DownloadResult:
    task: ReportTask
    ok: bool
    strategy: str
    reason: str = ""
    elapsed: float = 0.0
    retry_round: int = 0


def configure_logger() -> logging.Logger:
    ROOT_DIR.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("cafef")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    for handler in (logging.StreamHandler(), logging.FileHandler(LOG_FILE, encoding="utf-8")):
        handler.setFormatter(fmt)
        logger.addHandler(handler)
    return logger


LOG = logging.getLogger("cafef")


def task_key(ticker: str, year: str, period: str, filename: str, href: str, onclick: str) -> str:
    raw = "|".join((ticker, year, period, filename, href, onclick))
    return hashlib.sha1(raw.encode("utf-8", "ignore")).hexdigest()[:20]

def normalize_space(value: str) -> str:
    return re.sub(r"\s+", " ", value or "").strip()


def is_valid_document(path: Path) -> tuple[bool, str]:
    """Reject small files, HTML/error pages and non-PDF content."""
    try:
        if not path.is_file() or path.stat().st_size < MIN_FILE_BYTES:
            return False, "smaller than 10 KB"
        head = path.read_bytes()[:2048].lstrip().lower()
        if head.startswith(b"%pdf-"):
            return True, "pdf"
        if (b"<html" in head or b"<!doctype html" in head or b"access denied" in head or
                b"403 forbidden" in head or b"404 not found" in head):
            return False, "HTML/error response"
        # CafeF may provide Office documents for a few report rows; preserve old
        # acceptance behavior provided it is not an HTML error page.
        return True, "binary document"
    except OSError as exc:
        return False, str(exc)


def usable_existing_file(task: ReportTask) -> bool:
    for path in (task.local_path, task.drive_path):
        valid, _ = is_valid_document(path)
        if valid:
            return True
        if path.exists():
            try:
                path.unlink()
            except OSError:
                pass
    return False

def exchange_for(ticker: str, session: requests.Session) -> str:
    try:
        response = session.get(f"https://api.vietstock.vn/ta/getticker?ticker={ticker.upper()}", timeout=12)
        payload = response.json()
        raw = str(payload[0].get("Exchange", "")).lower() if payload else ""
        return "hose" if "hose" in raw else "hnx" if "hnx" in raw else "upcom" if "upcom" in raw else "hose"
    except Exception as exc:
        LOG.warning("%s exchange lookup failed (%s); use hose", ticker, exc)
        return "hose"


def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(total=3, connect=3, read=3, backoff_factor=0.7,
                  status_forcelist=(429, 500, 502, 503, 504), allowed_methods=frozenset(("GET", "HEAD")))
    adapter = HTTPAdapter(max_retries=retry, pool_connections=REQUEST_WORKERS * 2, pool_maxsize=REQUEST_WORKERS * 2)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": USER_AGENT, "Accept": "application/pdf,application/octet-stream,*/*;q=0.8"})
    return session


_REQUEST_LOCAL = threading.local()


def worker_session() -> requests.Session:
    """One persistent Session per HTTP worker: connection pool and cookies survive tasks."""
    if not hasattr(_REQUEST_LOCAL, "session"):
        _REQUEST_LOCAL.session = build_session()
    return _REQUEST_LOCAL.session


def make_driver(download_dir: Path) -> webdriver.Chrome:
    options = webdriver.ChromeOptions()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1440,1200")
    options.add_argument(f"--user-agent={USER_AGENT}")
    options.add_experimental_option("prefs", {
        "download.default_directory": str(download_dir.resolve()),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "plugins.always_open_pdf_externally": True,
        "safebrowsing.enabled": True,
    })
    driver = webdriver.Chrome(options=options)
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {"behavior": "allow", "downloadPath": str(download_dir.resolve())})
    driver.set_page_load_timeout(40)
    return driver

def classify_report(title: str, row_text: str) -> tuple[str, bool, str, str] | None:
    text = normalize_space(f"{title} {row_text}").lower()
    if not ("báo cáo" in text or "bao cao" in text or "bctc" in text):
        return None
    match = re.search(r"\b(2021|2022|2023|2024|2025)\b", text)
    if not match:
        return None
    year = match.group(1)
    period, quarterly = "CN", False
    if "kiểm toán" in text or "kiem toan" in text:
        period = "CN"
    elif re.search(r"quý\s*1|quy\s*1|\bq1\b", text):
        period, quarterly = "Q1", True
    elif re.search(r"quý\s*2|quy\s*2|\bq2\b|6\s*tháng|6\s*thang|bán niên|ban nien", text):
        period, quarterly = "Q2", True
    elif re.search(r"quý\s*3|quy\s*3|\bq3\b", text):
        period, quarterly = "Q3", True
    elif re.search(r"quý\s*4|quy\s*4|\bq4\b", text):
        period, quarterly = "Q4", True
    suffix_m = "_M" if any(x in text for x in ("mẹ", "me", "riêng", "rieng", "lẻ", "le")) else ""
    suffix_ss = "_SS" if any(x in text for x in ("soát xét", "soat xet", "đã soát", "da soat", "\bss\b")) else ""
    return year, quarterly, period, suffix_m + suffix_ss


def extract_tasks(driver: webdriver.Chrome, ticker: str, exchange: str) -> list[ReportTask]:
    url = f"https://cafef.vn/du-lieu/{exchange}/{ticker.lower()}-tai-lieu.chn"
    driver.get(url)
    WebDriverWait(driver, PAGE_WAIT_SECONDS).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tr")))
    tasks: list[ReportTask] = []
    seen: set[str] = set()
    for row in driver.find_elements(By.CSS_SELECTOR, "tr"):
        row_text = normalize_space(row.text)
        if not any(year in row_text for year in YEARS):
            continue
        cells = row.find_elements(By.CSS_SELECTOR, "td")
        title = normalize_space(cells[0].text if cells else row_text.split("\n")[0])
        parsed = classify_report(title, row_text)
        if not parsed:
            continue
        year, quarterly, period, suffix = parsed
        sub_path = str(Path(ticker) / year / period) if quarterly else str(Path(ticker) / year)
        filename = f"{ticker}_{year}_{period}{suffix}.pdf"  # original naming contract
        # Every anchor is deliberately emitted.  A first valid download wins for
        # identical output names; alternative anchors remain retry candidates.
        for index, anchor in enumerate(row.find_elements(By.TAG_NAME, "a")):
            href = anchor.get_attribute("href") or ""
            onclick = anchor.get_attribute("onclick") or ""
            outer = anchor.get_attribute("outerHTML") or ""
            if not (href or onclick or outer):
                continue
            ident = task_key(ticker, year, period, filename, href, onclick)
            if ident in seen:
                continue
            seen.add(ident)
            tasks.append(ReportTask(ticker, exchange, year, period, title, filename, sub_path,
                                    href, onclick, outer, row_text, url, index, ident))
    LOG.info("%s discovery: %d link tasks", ticker, len(tasks))
    return tasks

def save_cache(tasks: Iterable[ReportTask]) -> None:
    payload = {"version": 2, "saved_at": time.time(), "tasks": [asdict(t) for t in tasks]}
    tmp = CACHE_FILE.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(CACHE_FILE)


def load_cache() -> list[ReportTask]:
    try:
        data = json.loads(CACHE_FILE.read_text(encoding="utf-8"))
        return [ReportTask.from_dict(item) for item in data.get("tasks", [])]
    except (OSError, ValueError, TypeError) as exc:
        LOG.warning("Cannot load task cache: %s", exc)
        return []


def candidate_urls(task: ReportTask) -> list[str]:
    text = " ".join((task.href, task.onclick, task.outer_html))
    urls: list[str] = []
    if task.href and not task.href.lower().startswith("javascript"):
        urls.append(urljoin(task.source_url, task.href))
    # direct documents first; supports quoted/unquoted attributes and escaped text
    urls.extend(re.findall(r"https?://[^\s'\"<>\\]+?\.(?:pdf|docx?|xlsx?)(?:\?[^\s'\"<>\\]*)?", text, re.I))
    urls.extend(urljoin(task.source_url, value) for value in re.findall(r"[\"']([^\"']+\.(?:pdf|docx?|xlsx?)[^\"']*)[\"']", text, re.I))
    # Do not limit the numeric id length: CafeF has very long zero-prefixed ids.
    for ident in re.findall(r"(?<!\d)(\d{6,})(?!\d)", text):
        urls.append(f"https://cafef.vn/download/{ident}")
    return list(dict.fromkeys(urls))


def write_response(task: ReportTask, response: requests.Response) -> tuple[bool, str]:
    content_type = response.headers.get("Content-Type", "").lower()
    if response.status_code != 200:
        return False, f"HTTP {response.status_code}"
    if len(response.content) < MIN_FILE_BYTES:
        return False, "response smaller than 10 KB"
    if "text/html" in content_type or response.content[:2048].lower().find(b"<html") >= 0:
        return False, "HTML response"
    task.local_path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=".part-", dir=str(task.local_path.parent))
    try:
        with os.fdopen(fd, "wb") as handle:
            handle.write(response.content)
        tmp = Path(tmp_name)
        valid, reason = is_valid_document(tmp)
        if not valid:
            tmp.unlink(missing_ok=True)
            return False, reason
        tmp.replace(task.local_path)
        return True, "saved"
    except Exception as exc:
        Path(tmp_name).unlink(missing_ok=True)
        return False, str(exc)

def request_download(task: ReportTask, cookies: list[dict]) -> DownloadResult:
    started = time.monotonic()
    session = worker_session()
    for cookie in cookies:
        try:
            session.cookies.set(cookie["name"], cookie["value"], domain=cookie.get("domain"), path=cookie.get("path", "/"))
        except Exception:
            pass
    headers = {"Referer": task.source_url, "Accept": "application/pdf,application/octet-stream,*/*"}
    errors = []
    for url in candidate_urls(task):
        try:
            response = session.get(url, headers=headers, timeout=(12, 60), allow_redirects=True)
            ok, reason = write_response(task, response)
            if ok:
                return DownloadResult(task, True, "requests-session", f"{url}", time.monotonic() - started)
            errors.append(f"{url}: {reason}")
        except requests.RequestException as exc:
            errors.append(f"{url}: {type(exc).__name__}")
    return DownloadResult(task, False, "requests-session", "; ".join(errors[-3:]), time.monotonic() - started)


class SeleniumFallback:
    """A single bounded Chrome used only after concurrent requests fail."""
    def __init__(self) -> None:
        self.download_dir = Path(tempfile.mkdtemp(prefix="cafef-chrome-"))
        self.driver: Optional[webdriver.Chrome] = None

    def start(self) -> None:
        if self.driver is None:
            self.driver = make_driver(self.download_dir)

    def restart(self) -> None:
        self.close_driver()
        self.start()

    def close_driver(self) -> None:
        if self.driver:
            try: self.driver.quit()
            except Exception: pass
            self.driver = None

    def close(self) -> None:
        self.close_driver()
        shutil.rmtree(self.download_dir, ignore_errors=True)

    def clear_downloads(self) -> None:
        for item in self.download_dir.iterdir():
            try: item.unlink()
            except OSError: pass

    def completed_download(self, before: set[Path]) -> Optional[Path]:
        deadline = time.monotonic() + DOWNLOAD_WAIT_SECONDS
        while time.monotonic() < deadline:
            files = [p for p in self.download_dir.iterdir() if p not in before and not p.name.endswith(".crdownload")]
            if files:
                return max(files, key=lambda p: p.stat().st_mtime)
            time.sleep(0.25)
        return None

    def find_anchor(self, task: ReportTask):
        assert self.driver
        self.driver.get(task.source_url)
        WebDriverWait(self.driver, PAGE_WAIT_SECONDS).until(EC.presence_of_all_elements_located((By.TAG_NAME, "a")))
        for anchor in self.driver.find_elements(By.TAG_NAME, "a"):
            if ((anchor.get_attribute("href") or "") == task.href and
                    (anchor.get_attribute("onclick") or "") == task.onclick):
                return anchor
        anchors = self.driver.find_elements(By.TAG_NAME, "a")
        return anchors[task.link_index] if task.link_index < len(anchors) else None

    def accept_download(self, task: ReportTask, before: set[Path]) -> tuple[bool, str]:
        downloaded = self.completed_download(before)
        if not downloaded:
            return False, "no completed browser download"
        valid, reason = is_valid_document(downloaded)
        if not valid:
            downloaded.unlink(missing_ok=True)
            return False, reason
        task.local_path.parent.mkdir(parents=True, exist_ok=True)
        downloaded.replace(task.local_path)
        return True, "saved"

    def download(self, task: ReportTask, retry_round: int) -> DownloadResult:
        started = time.monotonic()
        errors = []
        try:
            self.start()
            assert self.driver
            anchor = self.find_anchor(task)
            if anchor is None:
                return DownloadResult(task, False, "selenium", "anchor not found", time.monotonic()-started, retry_round)
            strategies = [
                ("normal-click", lambda a: a.click()),
                ("javascript-click", lambda a: self.driver.execute_script("arguments[0].click()", a)),
                ("actionchains-click", lambda a: ActionChains(self.driver).move_to_element(a).click(a).perform()),
                ("enter", lambda a: a.send_keys(Keys.ENTER)),
                ("new-tab", lambda a: self.driver.execute_script("window.open(arguments[0].href || 'about:blank','_blank')", a)),
            ]
            for name, action in strategies:
                self.clear_downloads()
                before = set(self.download_dir.iterdir())
                try:
                    anchor = self.find_anchor(task)  # DOM can be stale after a click/navigation
                    if anchor is None: raise WebDriverException("anchor disappeared")
                    self.driver.execute_script("arguments[0].scrollIntoView({block:'center'})", anchor)
                    action(anchor)
                    ok, reason = self.accept_download(task, before)
                    if ok:
                        return DownloadResult(task, True, name, reason, time.monotonic()-started, retry_round)
                    errors.append(f"{name}: {reason}")
                except Exception as exc:
                    errors.append(f"{name}: {type(exc).__name__}")
            # Browser cookies may be newer than discovery cookies: attempt HTTP again.
            retry = request_download(task, self.driver.get_cookies())
            retry.strategy = "requests-after-selenium-cookie"
            retry.retry_round = retry_round
            if retry.ok:
                return retry
            errors.append(retry.reason)
        except Exception as exc:
            errors.append(f"driver: {type(exc).__name__}: {exc}")
        return DownloadResult(task, False, "selenium-fallback", "; ".join(errors[-5:]), time.monotonic()-started, retry_round)

def sync_to_drive() -> None:
    DRIVE_TARGET_DIR.mkdir(parents=True, exist_ok=True)
    # Python implementation works in Colab even when rsync is unavailable.
    for source in ROOT_DIR.rglob("*"):
        if not source.is_file() or source == CACHE_FILE or source == LOG_FILE:
            continue
        relative = source.relative_to(ROOT_DIR)
        destination = DRIVE_TARGET_DIR / relative
        if is_valid_document(destination)[0]:
            continue
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)


def discover_all() -> tuple[list[ReportTask], dict[str, list[dict]]]:
    session = build_session()
    work = Path(tempfile.mkdtemp(prefix="cafef-discovery-"))
    driver = None
    try:
        driver = make_driver(work)
        all_tasks, cookies = [], {}
        for ticker in BANK_TICKERS:
            exchange = exchange_for(ticker, session)
            try:
                tasks = extract_tasks(driver, ticker, exchange)
                all_tasks.extend(tasks)
                cookies[ticker] = driver.get_cookies()
            except Exception as exc:
                LOG.exception("%s discovery failed: %s", ticker, exc)
                cookies[ticker] = []
        # Preserve all link candidates in cache but remove byte-for-byte duplicate tasks.
        unique = list({task.task_id: task for task in all_tasks}.values())
        save_cache(unique)
        return unique, cookies
    finally:
        if driver:
            try: driver.quit()
            except Exception: pass
        shutil.rmtree(work, ignore_errors=True)

def run_pipeline() -> None:
    ROOT_DIR.mkdir(parents=True, exist_ok=True)
    cached = load_cache() if CACHE_FILE.exists() else []
    if cached:
        tasks, discovery_cookies = cached, defaultdict(list)
        LOG.info("Resume cache: %d report-link tasks", len(tasks))
    else:
        tasks, discovery_cookies = discover_all()
    pending = [task for task in tasks if not usable_existing_file(task)]
    LOG.info("Tasks=%d | already valid=%d | pending=%d", len(tasks), len(tasks)-len(pending), len(pending))
    stats = defaultdict(Counter)
    retry_queue: deque[ReportTask] = deque()
    # Stage 1: concurrent direct/session downloads. Each task owns its Session.
    with ThreadPoolExecutor(max_workers=REQUEST_WORKERS, thread_name_prefix="cafef-http") as pool:
        futures = {pool.submit(request_download, task, discovery_cookies.get(task.ticker, [])): task for task in pending}
        for future in as_completed(futures):
            result = future.result()
            stats[result.task.ticker]["found"] += 1
            if result.ok:
                stats[result.task.ticker]["downloaded"] += 1
                LOG.info("%s %s %s | %s | %.1fs", result.task.ticker, result.task.year, result.task.period, result.strategy, result.elapsed)
            else:
                retry_queue.append(result.task)
                LOG.warning("%s %s %s | queue retry | %s", result.task.ticker, result.task.year, result.task.period, result.reason)
    # Stage 2: one Chrome, full click strategy set, refresh/restart between retry rounds.
    browser = SeleniumFallback()
    try:
        for retry_round in range(1, MAX_RETRY_ROUNDS + 1):
            if not retry_queue:
                break
            round_tasks, retry_queue = list(retry_queue), deque()
            LOG.info("Retry round %d: %d tasks", retry_round, len(round_tasks))
            if retry_round > 1:
                browser.restart()
            for task in round_tasks:
                if usable_existing_file(task):
                    stats[task.ticker]["downloaded"] += 1
                    continue
                result = browser.download(task, retry_round)
                if result.ok:
                    stats[task.ticker]["downloaded"] += 1
                    stats[task.ticker]["retry_success"] += 1
                    LOG.info("%s %s %s | %s retry=%d | %.1fs", task.ticker, task.year, task.period, result.strategy, retry_round, result.elapsed)
                else:
                    retry_queue.append(task)
                    LOG.warning("%s %s %s | failed retry=%d | %s", task.ticker, task.year, task.period, retry_round, result.reason)
    finally:
        browser.close()
    # Validation pass uses the cached task set, so a Colab disconnect resumes precisely.
    unresolved = [task for task in tasks if not usable_existing_file(task)]
    for task in unresolved:
        stats[task.ticker]["failed"] += 1
    sync_to_drive()
    LOG.info("==================== SUMMARY ====================")
    for ticker in BANK_TICKERS:
        s = stats[ticker]
        LOG.info("%s | Found: %d | Downloaded: %d | Retry Success: %d | Failed: %d",
                 ticker, s["found"], s["downloaded"], s["retry_success"], s["failed"])
    if unresolved:
        LOG.error("Unresolved tasks remain in %s for next resume: %d", CACHE_FILE, len(unresolved))

def main() -> None:
    configure_logger()
    start = time.monotonic()

    run_pipeline()
    LOG.info("Completed in %.1f seconds", time.monotonic() - start)


if __name__ == "__main__":
    main()

In [ ]:
# ==============================================================================
# ĐỌC FILE EXCEL VÀ LƯU CÁC MÃ CHỨNG KHOÁN ĐÃ XUẤT HIỆN (DÙNG PANDAS)
# ==============================================================================
import json
from pathlib import Path
import pandas as pd

# ------------------------------------------------------------------------------
# CẤU HÌNH
# ------------------------------------------------------------------------------
EXCEL_PATH = Path("/content/drive/MyDrive/DanhMucTaiLieu.xlsx")
SHEET_NAME = "DanhMucTaiLieu"
COLUMN_NAME = "Mã Chứng Khoán"
OUTPUT_JSON = Path("/content/drive/MyDrive/Du_Lieu_BCTC_Cac_Cong_Ty/tickers_da_co.json")
OUTPUT_TXT  = Path("/content/drive/MyDrive/Du_Lieu_BCTC_Cac_Cong_Ty/tickers_da_co.txt")

# ------------------------------------------------------------------------------
def extract_tickers_pandas(excel_path: Path, sheet_name: str, column_name: str) -> list[str]:
    print(f"[*] Đang đọc file: {excel_path} ...")

    # Đọc file excel (mặc định pandas sẽ tự bỏ qua các dòng trống ở đầu nếu có)
    # Ta đọc toàn bộ sheet vào DataFrame
    df = pd.read_excel(excel_path, sheet_name=sheet_name)

    # 1. Tìm cột: Đôi khi header không nằm ở dòng đầu tiên của file Excel
    # Ta sẽ quét qua các cột hiện tại của df
    target_col = None
    for col in df.columns:
        if str(col).strip() == column_name:
            target_col = col
            break

    # Nếu không tìm thấy ở header chính, ta quét tiếp trong các hàng dữ liệu đầu tiên
    # (trường hợp header bị lệch xuống dưới vài dòng)
    if target_col is None:
        for idx, row in df.head(10).iterrows():
            for col in df.columns:
                if str(row[col]).strip() == column_name:
                    # Đặt lại tên cột và cắt bỏ phần thừa phía trên
                    df.columns = df.iloc[idx]
                    df = df.iloc[idx + 1:]
                    target_col = column_name
                    break
            if target_col:
                break

    if target_col is None:
        raise ValueError(f"Không tìm thấy cột '{column_name}' trong sheet '{sheet_name}'.")

    # 2. Lấy dữ liệu, chuẩn hóa (viết hoa, xóa khoảng trắng, loại bỏ NaN)
    print(f"[+] Đang trích xuất mã chứng khoán từ cột '{target_col}'...")
    tickers_series = df[target_col].dropna().astype(str)

    tickers = set()
    for val in tickers_series:
        cleaned = val.strip().upper()
        if cleaned and cleaned != "NAN": # Loại bỏ các giá trị rỗng/lỗi chuỗi
            tickers.add(cleaned)

    return sorted(list(tickers))

def save_results(tickers: list[str], json_path: Path, txt_path: Path) -> None:
    # Đảm bảo thư mục cha tồn tại
    json_path.parent.mkdir(parents=True, exist_ok=True)

    # Lưu JSON
    json_path.write_text(json.dumps(tickers, ensure_ascii=False, indent=2), encoding="utf-8")
    # Lưu TXT
    txt_path.write_text("\n".join(tickers), encoding="utf-8")

# ------------------------------------------------------------------------------
# THỰC THI
# ------------------------------------------------------------------------------
# ------------------------------------------------------------------------------
# THỰC THI (ĐÃ NÂNG CẤP: TÍCH LŨY DỒN DẬP - KHÔNG LO MẤT MÃ CŨ)
# ------------------------------------------------------------------------------
try:
    # 1. Đọc danh sách mã mới từ file Excel hiện tại
    excel_tickers = extract_tickers_pandas(EXCEL_PATH, SHEET_NAME, COLUMN_NAME)
    print(f"[*] Tìm thấy {len(excel_tickers)} mã trong file Excel.")

    # 2. Đọc danh sách mã đã tích lũy từ trước trong file JSON (nếu có)
    existing_tickers = set()
    if OUTPUT_JSON.exists():
        try:
            old_data = json.loads(OUTPUT_JSON.read_text(encoding="utf-8"))
            existing_tickers = set(str(t).strip().upper() for t in old_data)
            print(f"[*] Đã đọc {len(existing_tickers)} mã lịch sử từ file JSON cũ.")
        except Exception:
            print("[!] File JSON cũ bị lỗi hoặc trống. Sẽ khởi tạo mới.")

    # 3. Gộp hai danh sách lại với nhau (tự động loại bỏ trùng lặp)
    final_tickers = sorted(list(existing_tickers.union(excel_tickers)))

    # 4. Lưu lại kết quả cuối cùng
    print(f"\n[+] Tổng số mã sau khi gộp tích lũy: {len(final_tickers)}")
    save_results(final_tickers, OUTPUT_JSON, OUTPUT_TXT)
    print(f"[+] Đã cập nhật và lưu kết quả thành công!")

except FileNotFoundError:
    print(f"[-] Lỗi: Không tìm thấy file tại '{EXCEL_PATH}'.")
except Exception as e:
    print(f"[-] Đã xảy ra lỗi: {e}")

In [ ]:
# ==============================================================================
# BƯỚC 2: ĐƯA LOG DOWNLOAD THÀNH CÔNG LÊN GOOGLE SHEETS (Bản tìm tên thông minh)
# ==============================================================================

!pip install gspread > /dev/null

from google.colab import auth
import gspread
from google.auth import default

print("[*] Vui lòng cấp quyền truy cập Google Drive/Google Sheets trong popup hiện ra...")
auth.authenticate_user()

# --- COPY TOÀN BỘ CODE VÀO ĐÂY ---
from __future__ import annotations
import json
import unicodedata
from pathlib import Path
from typing import Any

SHEET_NAME = "DanhMucTaiLieu"
COLLECTOR_NAME = "Nguyễn Trần Bảo Thái"
HEADERS = (
    "Mã Chứng Khoán",
    "Tên Công ty",
    "Năm Báo Cáo",
    "Loại Báo Cáo",
    "Người Thu Thập",
    "Ghi Chú",
)

def _text(value: Any) -> str:
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    return str(value or "").strip()

def _normalize_name(name: str) -> str:
    """Loại bỏ dấu tiếng Việt, khoảng trắng thừa và chuyển thành chữ thường để so sánh."""
    name = str(name or "").strip().lower()
    return unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')

def load_success_log(log_path: Path) -> list[list[Any]]:
    rows: list[list[Any]] = []
    if not log_path.exists():
        return rows
    for line_number, line in enumerate(log_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip(): continue
        item = json.loads(line)
        missing = [header for header in HEADERS if header not in item]
        if missing:
            raise ValueError(f"Log row {line_number} is missing columns: {missing}")
        rows.append([item.get(header, "") for header in HEADERS])
    return rows

def _new_cell(value: Any) -> dict[str, Any]:
    if isinstance(value, bool): return {"userEnteredValue": {"boolValue": value}}
    if isinstance(value, (int, float)): return {"userEnteredValue": {"numberValue": value}}
    return {"userEnteredValue": {"stringValue": str(value)}}

def write_log_below_last_collector(
    spreadsheet_id: str,
    log_path: Path,
    *,
    collector_name: str = COLLECTOR_NAME,
    clear_log_after_commit: bool = True,
) -> int:
    pending_rows = load_success_log(log_path)
    if not pending_rows:
        return 0

    creds, _ = default()
    client = gspread.authorize(creds)
    worksheet = client.open_by_key(spreadsheet_id).worksheet(SHEET_NAME)

    existing_rows = worksheet.get("A2:F")
    existing_keys = {tuple(_text(value) for value in row[:6]) for row in existing_rows}

    rows_to_insert = [
        row for row in pending_rows
        if tuple(_text(value) for value in row) not in existing_keys
    ]
    if not rows_to_insert:
        if clear_log_after_commit:
            log_path.write_text("", encoding="utf-8")
        return 0

    # Cơ chế tìm tên thông minh mới
    normalized_collector = _normalize_name(collector_name)
    collector_rows = [
        index + 2 for index, row in enumerate(existing_rows)
        if len(row) >= 5 and _normalize_name(row[4]) == normalized_collector
    ]

    if not collector_rows:
        last_collector_row = len(existing_rows) + 1
    else:
        last_collector_row = max(collector_rows)

    insert_at = last_collector_row + 1
    row_count = len(rows_to_insert)

    requests = [
        {"insertDimension": {"range": {"sheetId": worksheet.id, "dimension": "ROWS", "startIndex": insert_at - 1, "endIndex": insert_at - 1 + row_count}, "inheritFromBefore": True}},
        {"copyPaste": {"source": {"sheetId": worksheet.id, "startRowIndex": last_collector_row - 1, "endRowIndex": last_collector_row, "startColumnIndex": 0, "endColumnIndex": 6}, "destination": {"sheetId": worksheet.id, "startRowIndex": insert_at - 1, "endRowIndex": insert_at - 1 + row_count, "startColumnIndex": 0, "endColumnIndex": 6}, "pasteType": "PASTE_FORMAT", "pasteOrientation": "NORMAL"}},
        {"updateCells": {"start": {"sheetId": worksheet.id, "rowIndex": insert_at - 1, "columnIndex": 0}, "rows": [{"values": [_new_cell(value) for value in row]} for row in rows_to_insert], "fields": "userEnteredValue"}}
    ]

    worksheet.spreadsheet.batch_update({"requests": requests})
    if clear_log_after_commit:
        log_path.write_text("", encoding="utf-8")

    return row_count

# --- ĐOẠN CODE THỰC THI ---

SPREADSHEET_ID = "Id_Sheet"
LOG_PATH = Path("./Du_Lieu_BCTC_Cac_Cong_Ty/successful_downloads.jsonl")

print("[*] Đang tìm kiếm tên và đẩy dữ liệu lên Google Sheets...")
try:
    inserted_rows = write_log_below_last_collector(
        spreadsheet_id=SPREADSHEET_ID,
        log_path=LOG_PATH
    )
    if inserted_rows > 0:
        print(f"[+] Hoàn tất! Đã chèn {inserted_rows} dòng báo cáo mới ngay dưới dòng cũ của bạn.")
    else:
        print("[!] Không có dữ liệu mới nào được thêm vào (Có thể log rỗng hoặc dữ liệu đã bị trùng).")
except Exception as e:
    print(f"[-] Đã xảy ra lỗi khi ghi dữ liệu: {e}")